# Replacing one kernel inside a process

`examples/replace_process.ipynb` fills a process slot: a whole compute block
answered by a replay or a network.  This notebook goes one level down.  Every
process freeCAM owns as a class also names the numerical kernels inside its
driver (`stage.kernels`), and each of those is a slot of its own:

- a **Python callable** at the kernel's pause: the driver runs in Fortran,
  pauses at the call, hands the arguments to Python by the kernel's own dummy
  names, writes the answer back and resumes (three crossings a call);
- a **compiled plugin at the hook**: a Python function compiled with Numba
  and bound inside the image, called by Fortran where the kernel was, with no
  Python in the step;
- the **original through the pause**: the same crossings with the original
  answering, which is how each slot is proved bit-for-bit.

What every kernel takes and gives is in `docs/contracts.md`; what each path
costs is in `docs/physics_kernel_decoupling.md`.

## 0. Does this checkout have what it needs?

Same preflight as the process notebook: the site file, the reference case, the image.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from freecam import site

where = site.resolved(repo=REPO)
for name in ('account', 'queue', 'scratch', 'reference case', 'reference run'):
    print(f'{name:<16}{where[name]}')
print()

checks = site.preflight(repo=REPO)
for check in checks:
    print(f'  {check}')
absent = [check for check in checks if not check.ok]
assert not absent, (
    '\n\nnot ready.  What is missing, and what produces it:\n  '
    + '\n  '.join(f'{check.name}: {check.produced_by}' for check in absent)
    + '\n\nSee the README, section "Site configuration".')

## 1. The kernels a process owns

Looking a process up binds nothing.  `kernels` maps each kernel the driver
calls to its slot (`None`: the original); `describe_kernels()` adds the
reviewed contract, the binding and the calls so far.

In [ ]:
import freecam as fc

STEPS = int(os.environ.get('PYCAM_STEPS', 4))

driver = fc.Driver(case='PI-atm', nsteps=STEPS, restart_every='end')
print('processes owned as classes:', ', '.join(driver.processes))

vdiff = driver.processes['vertical_diffusion']
deep = driver.processes['deep_convection']
for stage in (vdiff, deep):
    print()
    print(stage.STAGE)
    for row in stage.describe_kernels():
        contract = row.get('contract') or {}
        print(f"  {row['kernel']:<20} contract {contract.get('path') or '-'}")

## 2. The original through the pause

`OriginalKernel()` in a slot makes the driver pause at that call and the
original Fortran answer on the paused frame.  The state is bit-for-bit the
oracle's; the cost is the three crossings.  This is the gate every slot
passes before anything else stands in it.

In [ ]:
from freecam.physics.segments import OriginalKernel

vdiff.kernels['compute_tms'] = OriginalKernel()

driver.close()                      # release a model left over from a rerun
driver.initialize()                 # the 512-rank job: as long as the queue takes
print('run directory:', driver.run_dir)

result = driver.run(steps=STEPS, progress=True)
print('who computed vertical diffusion:', driver.status['processes']['vertical_diffusion'])

## 3. A Python function at the pause

A callable in the slot receives the kernel's arguments as NumPy arrays keyed
by the dummy names of its contract (the chunk's columns, `ncol` live) and
returns the outputs by name.  `cldfrc_fice_reference` is the ice-fraction
kernel written in NumPy, bit-for-bit with the Fortran; it stands in the deep
convection driver where `cldfrc_fice` is called.

In [ ]:
import importlib.util

source = REPO / 'examples/plugins/numba_kernels/cldfrc_fice.py'
spec = importlib.util.spec_from_file_location('cldfrc_fice_example', source)
example = importlib.util.module_from_spec(spec)
spec.loader.exec_module(example)


def ice_fraction(batch):
    fice, fsnow = example.cldfrc_fice_reference(batch['t'])
    return {'fice': fice, 'fsnow': fsnow}


deep.kernels['cldfrc_fice'] = ice_fraction
driver.advance(STEPS)
print('who computed deep convection:', driver.status['processes']['deep_convection'])

## 4. The same function compiled, called by Fortran

`compile_kernel` turns the function into a plugin the image binds at the
kernel's hook: Fortran calls it directly, no Python runs in the step, and
the stage runs whole.  The function takes the hook's model block -- inputs
then outputs, as arrays -- and fills the outputs in place.

In [ ]:
from freecam.physics.numba_kernel import compile_kernel

deep.kernels['cldfrc_fice'] = compile_kernel('cldfrc_fice', example.cldfrc_fice)
driver.advance(STEPS)
print('who computed deep convection:', driver.status['processes']['deep_convection'])

## 5. Back to the original

Empty slots detach the stage: the Fortran path is exactly the original again.

In [ ]:
deep.kernels['cldfrc_fice'] = None
vdiff.kernels['compute_tms'] = None
driver.advance(STEPS)
print(driver.status['processes'])

## 6. Release the allocation

In [ ]:
driver.close()

## Where the same thing is on the command line

`--segmented-original-kernels NAME[,NAME]` answers kernels with the original
through their pauses; `--kernel-plugin NAME=file.py:function` binds a
Numba-compiled function at a hook and `--shadow-kernel-plugin` runs it with
the original answering (the run stays bit-for-bit, the plugin's cost is
timed); `--kernel-model NAME=model.pt` binds a TorchScript file at a hook
through FTorch.  The 50-step and month jobs take them as
`PYCAM_ORIGINAL_KERNELS`, `PYCAM_KERNEL_PLUGINS`, `PYCAM_SHADOW_PLUGINS`
and `PYCAM_SHADOW_MODELS`.  The kernels that have a hook are listed in
`native/pi_cam/hooks.yaml`; the ones reached at a pause only, in the
runner manifest `native/pi_cam/segment_runners.yaml`.